In [1]:
import pandas as pd
import os

In [ ]:
## 두 개의 CSV 파일 수직으로 합치기

def merge_csv(path1, path2):
    root = 'data/original'
    
    print(os.path.join(root,' daily_observations_2008.csv'))
    # CSV 파일 읽기
    df1 = pd.read_csv(os.path.join(root, 'daily_observations_2007.csv'))
    df2 = pd.read_csv(os.path.join(root, 'daily_observations_2008.csv'))
    
    # 두 데이터프레임 수직 결합
    merged_df = pd.concat([df1, df2], ignore_index=True)
    
    # 결과 저장
    merged_df.to_csv(os.path.join(root, 'weather_original.csv'), index=False)
    
    print("CSV 파일 두 개가 수직으로 합쳐졌습니다.")

In [ ]:
root = 'data/original/preprocessing/'
path = os.path.join(root, '0_weather_original.csv')

df = pd.read_csv(path)
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])

# 30분 단위 -> 한시간 단위
df_filtered = df[df['Datetime'].dt.minute == 0]

# 필요없는 열 버리기
df_filtered = df_filtered.drop(columns=['Dew Point'])
df_filtered = df_filtered.drop(columns=['Wind'])
df_filtered = df_filtered.drop(columns=['Wind Speed'])
df_filtered = df_filtered.drop(columns=['Wind Gust'])
df_filtered = df_filtered.drop(columns=['Pressure'])
df_filtered = df_filtered.drop(columns=['Precip.'])
df_filtered = df_filtered.drop(columns=['Condition'])
df_filtered = df_filtered.drop(columns=['Date', 'Time'])

# DateTime 맨 앞으로
df_filtered = df_filtered[['Datetime'] + [col for col in df_filtered.columns if col != 'Datetime']]

#파일로 저장
output_file = '1_filtered_data.csv'
output_path = os.path.join(root, output_file)
df_filtered.to_csv(output_path, index=False)

print(df_filtered[:5])

In [ ]:
# household_power_consumption.csv 처리
root = 'data/original'
path = os.path.join(root, 'household_power_consumption.csv')

# 1. 'Date'와 'Time'을 합쳐 datetime 형식으로 변환
df = pd.read_csv(path)

# 숫자형 데이터로 변환 (문자열일 경우)
numeric_columns = ['Global_active_power', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')


# 'Date'와 'Time' 합쳐 datetime 변환
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')


# 2. 날짜 필터링
start_date = '2007-01-01'
end_date = '2010-07-01'
df = df[(df['Datetime'] >= start_date) & (df['Datetime'] <= end_date)]


# 3. Datetime을 시간 단위로 내림 처리
df['Datetime'] = df['Datetime'].dt.floor('H')

# 4. 필요 없는 열 제거
df = df.drop(columns=['Global_reactive_power', 'Voltage', 'Date', 'Time'])

# 5. 열 순서 재배치 (Datetime을 맨 앞으로 이동)
df = df[['Datetime'] + [col for col in df.columns if col != 'Datetime']]

# 6. 한 시간 단위로 값 합산
df = df.groupby('Datetime')[numeric_columns].sum().reset_index()

print(df[:5])


root = 'data/original/preprocessing/'
output_file = '2_aggregated_household_power_consumption.csv'
output_path = os.path.join(root, output_file)
df.to_csv(output_path, index=False)

In [21]:
root = 'data/original/preprocessing'
file1_path = os.path.join(root, '1_1_filtered_data.csv')
file2_path = os.path.join(root, '2_aggregated_household_power_consumption.csv')

df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# 첫 번째 열을 기준으로 비교 (가정: 첫 번째 열 이름이 'Datetime')
key_column = df1.columns[0]  # 첫 번째 열의 이름 가져오기
set1 = set(df1[key_column])
set2 = set(df2[key_column])

# 1. file1에만 있는 값
only_in_file1 = set1 - set2

# 2. file2에만 있는 값
only_in_file2 = set2 - set1

# 3. 결과 정리
missing_data = {
    'Only in file1': list(only_in_file1),
    'Only in file2': list(only_in_file2),
}

# 누락된 데이터를 DataFrame으로 변환
missing_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in missing_data.items()]))

# 결과 저장
output_path = os.path.join(root, 'missing_data.csv')
missing_df.to_csv(output_path, index=False)

print(f"누락된 데이터 비교 결과가 저장되었습니다: {output_path}")

누락된 데이터 비교 결과가 저장되었습니다: data/original/preprocessing/missing_data.csv


In [27]:
root = 'data/original/preprocessing'
file2_path = os.path.join(root, '1_1_filtered_data.csv')
file1_path = os.path.join(root, '2_aggregated_household_power_consumption.csv')

# 두 CSV 파일 읽기
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# 첫 번째 열을 기준으로 병합 (datetime 열이 첫 번째 열이라고 가정)
merged_df = pd.merge(df1, df2, on=df1.columns[0], how='outer', suffixes=('_file1', '_file2'))

# 정렬 (datetime 기준)
merged_df = merged_df.sort_values(by=df1.columns[0]).reset_index(drop=True)

# 결과 저장
output_path = os.path.join(root, '3_merged_data.csv')
merged_df.to_csv(output_path, index=False)


In [5]:
root = 'data/original/preprocessing'
file_path = os.path.join(root, '3_merged_data.csv')
df = pd.read_csv(file_path)

# 특정 열에서 불필요한 문자 제거
columns_to_clean = ['Temperature', 'Humidity']  # 처리할 열 이름
for col in columns_to_clean:
    df[col] = df[col].str.replace(r'[^\d.]+', '', regex=True).astype(float)  # 숫자와 '.'만 남기기

# 결과 저장
output_path = os.path.join(root, '4_final_data.csv')
df.to_csv(output_path, index=False)

print(f"불필요한 문자를 제거한 데이터가 저장되었습니다: {output_path}")


불필요한 문자를 제거한 데이터가 저장되었습니다: data/original/preprocessing/4_final_data.csv


In [5]:
# 1시간 -> 6시간 

import pandas as pd


def aggregate_to_6hour_intervals(file_path, output_file_path):
    # 데이터 로드 및 1행 제거
    data = pd.read_csv(file_path)
    data = data.iloc[1:]  # 첫 번째 행 제거

    # datetime 열을 datetime 형식으로 변환
    data['datetime'] = pd.to_datetime(data['datetime'])

    # datetime을 인덱스로 설정
    data.set_index('datetime', inplace=True)

    # 6시간 단위로 리샘플링
    resampled_data = data.resample('6H').agg({
        'Global_active_power': 'sum',
        'Global_intensity': 'sum',
        'Sub_metering_1': 'sum',
        'Sub_metering_2': 'sum',
        'Sub_metering_3': 'sum',
        'Temperature': ['min', 'max'],
        'Humidity': ['min', 'max']
    })

    # 컬럼명 변경
    resampled_data.columns = ['Global_active_power', 'Global_intensity', 'Sub_metering_1', 
                              'Sub_metering_2', 'Sub_metering_3', 'Temp_Min', 'Temp_Max', 
                              'Humidity_Min', 'Humidity_Max']

    # datetime 인덱스를 리샘플링된 시간의 마지막 시간으로 설정
    resampled_data.index = resampled_data.index + pd.Timedelta(hours=6)

    # 인덱스를 datetime 열로 변환
    resampled_data.reset_index(inplace=True)
    resampled_data.rename(columns={'index': 'datetime'}, inplace=True)

    # 결과를 CSV로 저장
    resampled_data.to_csv(output_file_path, index=False)

# 사용 예시
file_path = 'data/final_data.csv'
output_file_path = 'data/final_data_per_6hr.csv'
aggregate_to_6hour_intervals(file_path, output_file_path)

/tmp/ipykernel_530274/4051132165.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  resampled_data = data.resample('6H').agg({


In [8]:
# 6시간 데이터에 온습도 평균, range 정보 추가 


# Example usage
input_path = "data/final_data_per_6hr.csv"  # Replace with the path to your input file
output_path = "data/final_data_per_6hr_with_avg_range.csv"  # Replace with the desired output file path

# CSV 파일 읽기
df = pd.read_csv(input_path)

# 평균 및 Range 계산
if 'Temp_Min' in df.columns and 'Temp_Max' in df.columns:
    df['Temp_Avg'] = (df['Temp_Min'] + df['Temp_Max']) / 2
    df['Temp_Range'] = df['Temp_Max'] - df['Temp_Min']

if 'Humidity_Min' in df.columns and 'Humidity_Max' in df.columns:
    df['Humidity_Avg'] = (df['Humidity_Min'] + df['Humidity_Max']) / 2
    df['Humidity_Range'] = df['Humidity_Max'] - df['Humidity_Min']

# 컬럼 순서 지정
columns_order = [
    'datetime', 'Global_active_power', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
    'Temp_Min', 'Temp_Max', 'Temp_Avg', 'Temp_Range',
    'Humidity_Min', 'Humidity_Max', 'Humidity_Avg', 'Humidity_Range'
]

# 정렬된 컬럼 순서로 데이터프레임 재구성
df = df[columns_order]

# CSV 파일 저장
df.to_csv(output_path, index=False)

print(f"Processed data saved to {output_path}")


Processed data saved to data/final_data_per_6hr_with_avg_range.csv
